# End-to-end linear penalty model

In [2]:
# Allows Python to find mosh_src if this notebook is moved
import sys
sys.path.insert(0, "../")

# Add functionality to parse folders
import os

# Import matplotlib and numpy - helpful Python libraries for scientific computing
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.patches as patches
from matplotlib import ticker
from math import pi
from decimal import Decimal

# Import all MOSH plotting functions
from mosh_src.plotting_functions import *

### Step 1 - construct dose rate matrix

In [3]:
# Define parameters
dose_points = np.array([(0, 0.25, 0.75), (0, 0.4, 0.8), (0, 0.4, 0.4)]) # cm - note that these are the same across all output files
dwell_pos_transZ = np.array([-0.5, -0.4, -0.3, -0.2, -0.1, 0]) # cm - assuming source is centered in applicator (x, y) = (0, 0)
output_files = np.array(["rate_extract_test_5.csv", "rate_extract_test_4.csv", "rate_extract_test_3.csv", "rate_extract_test_2.csv", 
                        "rate_extract_test_1.csv", "rate_extract_test_0.csv"])

if np.shape(dwell_pos_transZ)[0] != np.shape(output_files)[0]:
    raise ValueError("Number of dwell position Z coordinates must match number of output files")

num_dwell_pos = np.shape(output_files)[0]
num_dose_points = np.shape(dose_points)[0]
source_dose_rate = 5 # Gy/s - same across all output files

# Number of bins and dosing grid half-length - both set in TOPAS parameter files
bins_mosh = 100
dose_grid_half_mosh = 5 # cm

# Computes the lattice length for the simulation space in cm
lattice_mosh = dose_grid_half_mosh*2/bins_mosh

In [ ]:
D = np.zeros((num_dose_points, num_dwell_pos))

for j in range(num_dwell_pos):
    # Step 1 - extract normalized dose rate @ source
    source_plane = cm_to_voxel(np.array(dwell_pos_transZ[j]), bins_mosh, lattice_mosh)
    source_energy, __ = extract_norm_energy(output_files[j], source_plane, 2, (0, 0), bins_mosh, lattice_mosh)

    # Step 2 - iterate over dose points, extract energy, and scale to source_norm_energy
    for i in range(num_dose_points):
        dose_point_transZ = dose_points[i][2]
        
        dose_plane = cm_to_voxel(np.array(dose_point_transZ), bins_mosh, lattice_mosh)
        dose_energy, __ = extract_norm_energy(output_files[j], dose_plane, 2, (dose_points[i][0], dose_points[i][1]), bins_mosh, lattice_mosh)

        normed_dose_energy = (dose_energy/source_energy)*source_dose_rate
        D[i][j] = normed_dose_energy

print(D)

### Step 2 - run linear penalty model

In [4]:
# Define basic optimization parameters
D_min = 2.73 # Gy
D_max = 10 # Gy
M_l = 1 # penalty/Gy
M_u = 1 # penalty/Gy

In [ ]:
# Construct the bounds A_ub and b_ub
# Step 1 - construct A matrix for dwell time vector (size 2r x c)
A_t = np.vstack((-M_l*D[0,:], M_u*D[0,:]))
for j in range(1, num_dose_points):
    A_t = np.vstack((A_t, -M_l*D[j,:], M_u*D[j,:]))
print(A_t)

# Step 2 - construct A matrix for dose point vector (size 2r x r)
A_w = -1*np.identity(num_dose_points)
for j in range(0, num_dose_points):
    A_w = np.insert(A_w, 2*j, A_w[2*j,:], axis=0)
print(A_w)

# Step 3 - combine to create A_ub
A_ub = np.hstack((A_t, A_w))
print(A_ub)

# Step 4 - construct b_ub
b_ub = np.reshape(np.full((num_dose_points,2), [-M_l*D_min, M_u*D_max]), 2*num_dose_points)
print(b_ub)

In [ ]:
# Construct the parameter vector c
# Number of 0 entries = number of dwell positions
# Number of 1 entries = number of dose points
c = np.concatenate((np.zeros(num_dwell_pos), np.ones(num_dose_points)))
print(c)

In [ ]:
# Set up the optimization
from scipy.optimize import linprog
res = linprog(c, A_ub=A_ub, b_ub=b_ub)
print(res.fun)
print(res.x)
res.message